# Tratamento dos preços do petróleo e do câmbio.

In [112]:
exchange_data = pd.read_csv('./files/brl_usd_exchange.csv')
oil_prices = pd.read_csv('./files/oil_prices.csv')

exchange_data = exchange_data.rename(columns={'DEXBZUS': 'price'})
oil_prices = oil_prices.rename(columns={'DCOILBRENTEU': 'price'})

# converter para datetime
exchange_data['date'] = pd.to_datetime(exchange_data['observation_date'])
oil_prices['date'] = pd.to_datetime(oil_prices['observation_date'])

exchange_data = exchange_data.drop('observation_date', axis=1)
oil_prices = oil_prices.drop('observation_date', axis=1)

#------------- adicionando colunas auxiliares
exchange_data['year'] = exchange_data['date'].dt.year
exchange_data['month'] = exchange_data['date'].dt.month

oil_prices['year'] = oil_prices['date'].dt.year
oil_prices['month'] = oil_prices['date'].dt.month
#-------------

# agregação mensal
exchange_data = (
    exchange_data
    .groupby(['year', 'month'])['price']
    .mean()
    .reset_index()
    .assign(date=lambda x: f"{x['year']}-{x['month']}")
)

exchange_data["date"] = (
    exchange_data["year"].astype(str) +
    "-" +
    exchange_data["month"].astype(str).str.zfill(2)
)

oil_prices = (
    oil_prices
    .groupby(['year', 'month'])['price']
    .mean()
    .reset_index()
)

oil_prices["date"] = (
    oil_prices["year"].astype(str) +
    "-" +
    oil_prices["month"].astype(str).str.zfill(2)
)

exchange_data.head(5)

,year,month,price,date
0,1995,1,0.846091,1995-01
1,1995,2,0.841150,1995-02
2,1995,3,0.890522,1995-03
3,1995,4,0.907400,1995-04
4,1995,5,0.897500,1995-05


# Tratamento dos dados de conflitos no Oriente Médio

In [160]:
import pandas as pd


conflicts = pd.read_excel('./files/middle_east_conflict.xls')
conflicts.info()

<class 'pandas.DataFrame'>
RangeIndex: 1957 entries, 0 to 1956
Data columns (total 26 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   ID          1957 non-null   int64         
 1   Location    1957 non-null   str           
 2   SideA       1957 non-null   str           
 3   SideA2nd    203 non-null    str           
 4   SideB       1957 non-null   str           
 5   SideB2nd    80 non-null     str           
 6   Incomp      1957 non-null   int64         
 7   Terr        1122 non-null   str           
 8   YEAR        1957 non-null   int64         
 9   Int         1957 non-null   int64         
 10  CumInt      1957 non-null   int64         
 11  Type        1957 non-null   int64         
 12  StartDate   1957 non-null   datetime64[us]
 13  StartPrec   1957 non-null   int64         
 14  StartDate2  1957 non-null   datetime64[us]
 15  Startprec2  1957 non-null   int64         
 16  EpEnd       1957 non-null   int64  

In [11]:
conflicts['SideB2nd'].value_counts()

SideB2nd
Democratic Republic of Vietnam                                                                                                                                                                                                                                   18
Australia, New Zealand, Philippines, Republic of Korea , Thailand, United States of America                                                                                                                                                                      11
Rwanda, Uganda                                                                                                                                                                                                                                                    4
Libya                                                                                                                                                                                                              

### carregando csv com países do Oriente Médio

In [161]:
middle_east_countries = pd.read_csv('./files/helper/middle_east_countries.csv')
middle_east_countries.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   country  5 non-null      str  
dtypes: str(1)
memory usage: 172.0 bytes


In [162]:
countries = set(middle_east_countries['country'])
cols = ['SideA', 'SideB', 'SideA2nd', 'SideB2nd']

def has_middle_east_country(cell):
    if pd.isna(cell):
        return False
    
    countries_cell = [p.strip() for p in str(cell).split(",")]
    return any(p in countries for p in countries_cell)

mask = conflicts[cols].apply(lambda col: col.map(has_middle_east_country)).any(axis=1)

middle_east_conflicts = conflicts[mask]
middle_east_conflicts.head(2)

,ID,Location,SideA,SideA2nd,SideB,SideB2nd,Incomp,Terr,YEAR,Int,...,EpEnd,EpEndDate,EpEndPrec,GWNOA,GWNOA2nd,GWNOB,GWNOB2nd,GWNOLoc,Region,Version
23,6,Iran,Iran,NaN,KDPI,Russia (Soviet Union),1,Kurdistan,1946,1,...,1,1946-12-16,-99.0,630,NaN,NaN,365,630,2,NaN
24,6,Iran,Iran,NaN,KDPI,NaN,1,Kurdistan,1966,1,...,0,NaT,NaN,630,NaN,NaN,NaN,630,2,NaN


In [146]:
middle_east_conflicts.info()

<class 'pandas.DataFrame'>
Index: 113 entries, 23 to 1904
Data columns (total 26 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   ID          113 non-null    int64         
 1   Location    113 non-null    str           
 2   SideA       113 non-null    str           
 3   SideA2nd    13 non-null     str           
 4   SideB       113 non-null    str           
 5   SideB2nd    7 non-null      str           
 6   Incomp      113 non-null    int64         
 7   Terr        67 non-null     str           
 8   YEAR        113 non-null    int64         
 9   Int         113 non-null    int64         
 10  CumInt      113 non-null    int64         
 11  Type        113 non-null    int64         
 12  StartDate   113 non-null    datetime64[us]
 13  StartPrec   113 non-null    int64         
 14  StartDate2  113 non-null    datetime64[us]
 15  Startprec2  113 non-null    int64         
 16  EpEnd       113 non-null    int64       

In [ ]:
# middle_east_conflicts["StartDate"] = pd.to_datetime(middle_east_conflicts["StartDate"])
# middle_east_conflicts["EpEndDate"] = pd.to_datetime(middle_east_conflicts["EpEndDate"])

# last_event = (
#     middle_east_conflicts
#     .groupby("ID")["EpEndDate"]
#     .transform("max")
# )

# middle_east_conflicts["EpEndDate"] = middle_east_conflicts["EpEndDate"].fillna(last_event)
# middle_east_conflicts["EpEndDate"] = middle_east_conflicts["EpEndDate"].clip(upper=pd.Timestamp("2010-12-31"))

In [163]:
conflict_periods = middle_east_conflicts[
    middle_east_conflicts["EpEndDate"].notna()
][["ID", "StartDate", "EpEndDate"]].copy()

conflict_periods["StartDate"] = pd.to_datetime(conflict_periods["StartDate"])
conflict_periods["EpEndDate"] = pd.to_datetime(conflict_periods["EpEndDate"])

conflict_periods["EpEndDate"] = conflict_periods["EpEndDate"].clip(
    upper=pd.Timestamp("2010-12-31")
)

In [38]:
middle_east_conflicts.columns

Index(['ID', 'Location', 'SideA', 'SideA2nd', 'SideB', 'SideB2nd', 'Incomp',
       'Terr', 'YEAR', 'Int', 'CumInt', 'Type', 'StartDate', 'StartPrec',
       'StartDate2', 'Startprec2', 'EpEnd', 'EpEndDate', 'EpEndPrec', 'GWNOA',
       'GWNOA2nd', 'GWNOB', 'GWNOB2nd', 'GWNOLoc', 'Region', 'Version'],
      dtype='str')

In [132]:
middle_east_conflicts['StartDate'].sort_values(ascending=False)

1904   2003-03-20
1895   2001-09-11
1896   2001-09-11
1897   2001-09-11
1898   2001-09-11
          ...    
38     1946-05-01
39     1946-05-01
24     1946-05-01
23     1946-05-01
40     1945-11-19
Name: StartDate, Length: 237, dtype: datetime64[us]

# Unindo datasets

In [70]:
exchange_data.head(5)

,year,month,price,date
0,1995,1,0.846091,1995-01
1,1995,2,0.841150,1995-02
2,1995,3,0.890522,1995-03
3,1995,4,0.907400,1995-04
4,1995,5,0.897500,1995-05


In [164]:
df = pd.merge(
    exchange_data,
    oil_prices,
    on="date",
    how="inner"
)

df = df.drop(['year_y', 'month_y'], axis=1).rename(columns={
    "price_x": "brl_usd_price", 
    "price_y": "oil_price",
    "year_x": "year",
    "month_x": "month"
})
df['date'] = pd.to_datetime(df['date'])
df.head(5)

,year,month,brl_usd_price,date,oil_price
0,1995,1,0.846091,1995-01-01,16.551905
1,1995,2,0.841150,1995-02-01,17.114500
2,1995,3,0.890522,1995-03-01,17.006522
3,1995,4,0.907400,1995-04-01,18.648333
4,1995,5,0.897500,1995-05-01,18.350909


In [165]:
# def has_conflict(month_start, month_end, conflicts):
#     active = conflicts[
#         (conflicts["StartDate"] <= month_end) &
#         (conflicts["EpEndDate"] >= month_start)
#     ]

#     return 1 if len(active) > 0 else 0

# conflict_list = []
# for date in df["date"]:
#     month_start = date.replace(day=1)
#     month_end = date

#     conflict = has_conflict(month_start, month_end, conflict_periods)
#     conflict_list.append(conflict)

# df["conflict"] = conflict_list


def count_conflicts(month_start, month_end, conflicts):
    active = conflicts[
        (conflicts["StartDate"] <= month_end) &
        (conflicts["EpEndDate"] >= month_start)
    ]

    return active["ID"].nunique()


conflict_list = []

for date in df["date"]:
    month_start = date.replace(day=1)
    month_end = date

    conflict = count_conflicts(month_start, month_end, middle_east_conflicts)
    conflict_list.append(conflict)

df["conflict_intensity"] = conflict_list

In [167]:
df['conflict_intensity'].value_counts()

conflict_intensity
0    96
1    61
4    24
Name: count, dtype: int64

In [169]:
df[df['conflict_intensity'] == 1].head(30)

,year,month,brl_usd_price,date,oil_price,conflict_intensity
24,1997,1,1.042552,1997-01-01,23.537619,1
25,1997,2,1.048789,1997-02-01,20.850526,1
26,1997,3,1.057005,1997-03-01,19.133000,1
27,1997,4,1.061005,1997-04-01,17.555909,1
28,1997,5,1.068000,1997-05-01,19.022857,1
29,1997,6,1.074614,1997-06-01,17.580000,1
30,1997,7,1.080477,1997-07-01,18.464500,1
31,1997,8,1.087862,1997-08-01,18.595500,1
32,1997,9,1.093548,1997-09-01,18.460952,1
33,1997,10,1.099895,1997-10-01,19.865217,1


In [154]:
df[df['conflict'] == 0].head(50)

,year,month,brl_usd_price,date,oil_price,conflict
84,2002,1,2.379905,2002-01-01,19.416818,0
85,2002,2,2.424189,2002-02-01,20.275500,0
86,2002,3,2.345024,2002-03-01,23.696667,0
87,2002,4,2.322682,2002-04-01,25.728636,0
88,2002,5,2.475250,2002-05-01,25.345455,0
89,2002,6,2.714375,2002-06-01,24.081667,0
90,2002,7,2.941409,2002-07-01,25.736087,0
91,2002,8,3.108159,2002-08-01,26.651364,0
92,2002,9,3.354775,2002-09-01,28.399524,0
93,2002,10,3.796614,2002-10-01,27.543043,0


In [102]:
df.dropna().info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   year           180 non-null    int32         
 1   month          180 non-null    int32         
 2   brl_usd_price  180 non-null    float64       
 3   date           180 non-null    datetime64[us]
 4   oil_price      180 non-null    float64       
 5   conflict       180 non-null    int64         
dtypes: datetime64[us](1), float64(2), int32(2), int64(1)
memory usage: 7.2 KB


In [170]:
df.tail(30)

,year,month,brl_usd_price,date,oil_price,conflict_intensity
151,2007,8,1.961952,2007-08-01,70.760870,0
152,2007,9,1.902290,2007-09-01,77.173158,0
153,2007,10,1.798704,2007-10-01,82.340000,0
154,2007,11,1.766881,2007-11-01,92.414286,0
155,2007,12,1.785175,2007-12-01,90.926842,0
156,2008,1,1.770962,2008-01-01,92.178095,0
157,2008,2,1.728995,2008-02-01,94.986500,0
158,2008,3,1.709048,2008-03-01,103.635500,0
159,2008,4,1.686345,2008-04-01,109.071364,0
160,2008,5,1.658462,2008-05-01,122.797143,0


In [155]:
df.to_csv('./files/results/final_dataframe.csv', index=False)